# Claim Denial — Scoring & LLM Explanations (Part 2)

Loads the **saved model** and **SHAP explainer** produced by `model_training.ipynb`,
scores `current_claims.csv`, and writes **`predictions_current_claims.csv`** sorted
from highest to lowest `denial_probability`, with columns:

`claim_id, denial_probability, predicted_denial, risk_tier, top_risk_factors, explanation`

Explanations for the **top-10 highest-risk** claims are written by an OpenAI LLM,
grounded **only** in each claim's field values and the model's SHAP risk drivers.

> **Prereq:** run `model_training.ipynb` once so the `artifacts/` folder exists.

In [1]:
#!pip install openai shap joblib

In [4]:
import os, csv, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import shap

## 1. Load the saved model + explainability objects

In [13]:
root_path = Path(r'C:\Users\shiva\Desktop\Ensemble_Partner_Hiring_Assignment\Hiring Assessment - AI Team - Classical ML + Gen AI problem (ML + basic Gen AI) (1)\Hiring Assessment - AI Team - Classical ML + Gen AI problem')
data_path = root_path / 'data'
art_path  = root_path / 'artifacts'


# The saved XGBoost model was tuned with this custom early-stopping metric, so its
def recall_at_25_loss(y_true, y_pred):
    y_true = np.asarray(y_true)
    n_review = max(1, int(np.ceil(len(y_pred) * 0.25)))
    top = np.argsort(y_pred)[::-1][:n_review]
    total_pos = y_true.sum()
    recall = y_true[top].sum() / total_pos if total_pos else 0.0
    return 1.0 - recall


# The model object + operating-point/schema metadata saved by the training notebook.
final_model = joblib.load(art_path / 'model.joblib')
meta        = joblib.load(art_path / 'metadata.joblib')

THRESHOLD  = meta['THRESHOLD']
MEDIAN_CUT = meta['MEDIAN_CUT']
FEATURES   = meta['FEATURES']
CATEGORICAL, BINARY_FLAGS = meta['CATEGORICAL'], meta['BINARY_FLAGS']
NUMERIC, ENGINEERED       = meta['NUMERIC'], meta['ENGINEERED']
feat_names = meta['feat_names']

prep = final_model.named_steps['prep']
clf  = final_model.named_steps['clf']

# The explainability object. Load the saved one; if pickle is unavailable,
try:
    explainer = joblib.load(art_path / 'shap_explainer.joblib')
except Exception as e:
    print('Rebuilding explainer from model:', e)
    explainer = shap.TreeExplainer(clf)
base_value = float(np.ravel(explainer.expected_value)[-1])

print(f'Loaded model + explainer.  THRESHOLD={THRESHOLD:.3f}  MEDIAN_CUT={MEDIAN_CUT:.3f}')

Loaded model + explainer.  THRESHOLD=0.579  MEDIAN_CUT=0.418


## 2. Load and score `current_claims.csv`

The **same** feature-engineering function used in training is applied so the
current claims get exactly the columns the model expects.

In [6]:
ID_COL = 'claim_id'

def engineer(df):
    df = df.copy()
    df['prior_auth_gap'] = ((df['prior_auth_required'] == 1) & (df['has_prior_auth'] == 0)).astype(int)
    df['referral_gap']   = ((df['referral_required'] == 1) & (df['referral_present'] == 0)).astype(int)
    df['payment_ratio']  = df['expected_payment'] / df['total_billed'].replace(0, np.nan)
    df['payment_ratio']  = df['payment_ratio'].fillna(0)
    df['log_total_billed'] = np.log1p(df['total_billed'])
    return df

curr   = pd.read_csv(data_path / 'current_claims.csv')
curr_e = engineer(curr)
curr_X = curr_e[FEATURES]

# ---- Score ------------------------------------------------------------------
denial_probability = final_model.predict_proba(curr_X)[:, 1]
predicted_denial   = (denial_probability >= THRESHOLD).astype(int)

def to_tier(p):
    if p >= THRESHOLD:  return 'High'
    if p >= MEDIAN_CUT: return 'Medium'
    return 'Low'
risk_tier = [to_tier(p) for p in denial_probability]

print(f'Scored {len(curr_e)} claims.  flagged (High) = {int(predicted_denial.sum())} '
      f'({predicted_denial.mean():.0%})')

Scored 500 claims.  flagged (High) = 115 (23%)


## 3. Per-claim risk factors via SHAP

SHAP runs on the fitted booster in the transformed space; one-hot contributions
are summed back to the original feature, so `top_risk_factors` names the real
claim fields. We keep the top 2-3 drivers by absolute contribution.

In [7]:
def _dense(X): return X.toarray() if hasattr(X, 'toarray') else np.asarray(X)

Xt_all = _dense(prep.transform(curr_X))
sv_all = explainer.shap_values(Xt_all)
if isinstance(sv_all, list):
    sv_all = sv_all[1]

def to_orig(feat):
    if feat.startswith('cat__'):
        rest = feat[len('cat__'):]
        for c in CATEGORICAL:
            if rest.startswith(c + '_'):
                return c
        return rest
    for pfx in ('num__', 'flag__', 'remainder__'):
        if feat.startswith(pfx):
            return feat[len(pfx):]
    return feat
orig_of = np.array([to_orig(f) for f in feat_names])

INT_COLS = set(BINARY_FLAGS) | {'prior_auth_gap', 'referral_gap',
                                'num_procedures', 'num_diagnoses', 'days_to_submit'}
def fmt_val(col, row):
    v = row[col]
    if col in CATEGORICAL: return f'{v}'
    if col in INT_COLS:    return f'{int(v)}'
    return f'{v:,.2f}'

def claim_drivers(pos, k=3):
    """Top-k original-feature drivers for one claim, ranked by |SHAP|."""
    agg = {}
    for f, s in zip(orig_of, sv_all[pos]):
        agg[f] = agg.get(f, 0.0) + float(s)
    ranked = sorted(agg.items(), key=lambda kv: abs(kv[1]), reverse=True)[:k]
    row = curr_e.iloc[pos]
    return [{'feature': f, 'value': fmt_val(f, row), 'shap': round(s, 3),
             'direction': 'raises' if s > 0 else 'lowers'} for f, s in ranked]

## 4. Assemble the prediction records (sorted highest → lowest risk)

In [8]:
records = []
for pos in range(len(curr_e)):
    drivers = claim_drivers(pos, k=3)
    trf = '; '.join(f"{d['feature']}={d['value']} ({d['direction']} risk)" for d in drivers)
    records.append({
        'claim_id':           curr_e.iloc[pos][ID_COL],
        'denial_probability': round(float(denial_probability[pos]), 4),
        'predicted_denial':   int(predicted_denial[pos]),
        'risk_tier':          risk_tier[pos],
        'top_risk_factors':   trf,
        'explanation':        '',      # filled below
        '_drivers':           drivers, # internal, dropped before CSV
    })

# Highest -> lowest denial probability.
records.sort(key=lambda r: r['denial_probability'], reverse=True)

print('Top 5 by risk:')
for r in records[:5]:
    print(f"  {r['claim_id']}  p={r['denial_probability']:.0%}  {r['risk_tier']:6s} | {r['top_risk_factors']}")

Top 5 by risk:
  CCLM-00285  p=86%  High   | prior_auth_gap=1 (raises risk); missing_documentation_flag=1 (raises risk); total_billed=71,797.00 (raises risk)
  CCLM-00062  p=85%  High   | prior_auth_gap=1 (raises risk); eligibility_verified=0 (raises risk); payment_ratio=0.33 (raises risk)
  CCLM-00352  p=84%  High   | prior_auth_gap=1 (raises risk); missing_documentation_flag=1 (raises risk); payment_ratio=0.31 (raises risk)
  CCLM-00013  p=83%  High   | prior_auth_gap=1 (raises risk); eligibility_verified=0 (raises risk); payment_ratio=0.32 (raises risk)
  CCLM-00087  p=83%  High   | missing_documentation_flag=1 (raises risk); days_to_submit=40 (raises risk); eligibility_verified=0 (raises risk)


## 5. LLM explanations (OpenAI) — prompt design

**Prompt template.** The system message pins the rules from the brief (grounded
only in provided facts, plain language, one action, hedge that it's an estimate,
2-3 sentences). The user message provides *only* the claim's probability, tier,
and SHAP drivers — nothing the model could hallucinate from.

**API key:** set it in environment **before** running — do **not** paste it
into the notebook. In a terminal: `setx OPENAI_API_KEY "sk-..."` then restart the
kernel. If no key is present, the notebook falls back to a deterministic template
so it still runs end-to-end.

In [ ]:
USE_LLM   = bool(os.environ.get('OPENAI_API_KEY'))
MODEL_NAME = 'gpt-4o-mini'

SYSTEM_PROMPT = (
    'You are a claims-denial analyst assistant. Write a short, plain-English risk '
    'explanation for ONE insurance claim. Follow every rule:\n'
    '- Use ONLY the facts provided (probability, tier, and risk drivers). Do NOT invent facts, '
    'codes, names, or dollar amounts that are not given.\n'
    '- Plain language an analyst can act on in seconds; no jargon or acronyms.\n'
    '- Exactly 2-3 sentences.\n'
    '- Include exactly ONE specific, concrete recommended action.\n'
    '- Make clear this is a model risk ESTIMATE, not a guarantee of denial.'
)

def build_user_prompt(rec):
    drivers = '\n'.join(
        f"- {d['feature']} = {d['value']} ({d['direction']} denial risk)"
        for d in rec['_drivers'])
    return (
        f"Claim ID: {rec['claim_id']}\n"
        f"Model denial probability: {rec['denial_probability']:.0%}\n"
        f"Risk tier: {rec['risk_tier']}\n"
        f"Top risk drivers (from the model):\n{drivers}\n\n"
        'Write the explanation now.'
    )

def llm_explain(rec):
    from openai import OpenAI
    client = OpenAI()  # reads OPENAI_API_KEY from the environment
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user',   'content': build_user_prompt(rec)}],
        temperature=0.2, max_tokens=160,
    )
    return resp.choices[0].message.content.strip()

def template_explanation(rec):
    """Deterministic fallback (no API needed), obeys the same content rules."""
    d0 = rec['_drivers'][0]
    return (f"This claim has an estimated {rec['denial_probability']:.0%} chance of denial "
            f"({rec['risk_tier'].lower()} risk), driven mainly by {d0['feature']} = {d0['value']}. "
            f"Recommended action: verify this field before submission. "
            f"This is a model estimate, not a guaranteed outcome.")

print('USE_LLM =', USE_LLM, '| model =', MODEL_NAME)

USE_LLM = True | model = gpt-4o-mini


## 6. Generate explanations for the top-10 riskiest claims

The LLM writes the top-10; the remaining claims get the deterministic template
(the brief only asks for LLM output on the top-10). Every row still ends up with
a populated `explanation` so the CSV is complete.

In [10]:
TOP_N = 10

for i, rec in enumerate(records[:TOP_N], 1):
    if USE_LLM:
        try:
            rec['explanation'] = llm_explain(rec)
        except Exception as e:
            print(f"  LLM failed for {rec['claim_id']} ({e}); using template.")
            rec['explanation'] = template_explanation(rec)
    else:
        rec['explanation'] = template_explanation(rec)
    print(f"[{i:2d}] {rec['claim_id']}  p={rec['denial_probability']:.0%}  {rec['risk_tier']}")
    print('    ', rec['explanation'], '\n')

# Remaining claims: deterministic template (LLM reserved for the top-10).
for rec in records[TOP_N:]:
    rec['explanation'] = template_explanation(rec)

[ 1] CCLM-00285  p=86%  High
     This claim has a high risk of denial, with an estimated 86% probability based on factors such as a gap in prior authorization and missing documentation. To improve the chances of approval, it is recommended to gather and submit any missing documents as soon as possible. Please note that this is an estimate and not a guarantee of denial. 

[ 2] CCLM-00062  p=85%  High
     This claim has a high risk of denial, with an estimated 85% probability based on several factors. The prior authorization gap, lack of verified eligibility, and low payment ratio all contribute to this risk. To improve the chances of approval, ensure that eligibility is verified before resubmitting the claim. 

[ 3] CCLM-00352  p=84%  High
     This claim has a high risk of denial, with an estimated 84% probability based on several factors. The claim is affected by a gap in prior authorization, missing documentation, and a low payment ratio. To improve the chances of approval, ensure 

## 7. Write `predictions_current_claims.csv`

Written with the stdlib `csv` writer (no pandas dependency for output), sorted
highest → lowest `denial_probability`.

In [11]:
out_path = root_path / 'predictions_current_claims.csv'
cols = ['claim_id', 'denial_probability', 'predicted_denial',
        'risk_tier', 'top_risk_factors', 'explanation']

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    for rec in records:                 # already sorted highest -> lowest
        w.writerow({c: rec[c] for c in cols})

print(f'Wrote {out_path}')
print(f'  {len(records)} rows, sorted by denial_probability (desc)')
print(f'  tiers: ' + ', '.join(f'{t}={sum(1 for r in records if r["risk_tier"]==t)}'
                               for t in ['High', 'Medium', 'Low']))

Wrote C:\Users\shiva\Desktop\Ensemble_Partner_Hiring_Assignment\Hiring Assessment - AI Team - Classical ML + Gen AI problem (ML + basic Gen AI) (1)\Hiring Assessment - AI Team - Classical ML + Gen AI problem\predictions_current_claims.csv
  500 rows, sorted by denial_probability (desc)
  tiers: High=115, Medium=139, Low=246


## 8. Sanity check — does it behave on a LOW-risk claim?

Same prompt, lowest-risk claim in the file. A well-designed prompt should produce
a calm, low-urgency message (not alarmist) and still hedge that it's an estimate.

In [12]:
low = records[-1]
print(f"Lowest-risk claim: {low['claim_id']}  p={low['denial_probability']:.0%}  {low['risk_tier']}")
print('Drivers:', low['top_risk_factors'])
print('\n--- PROMPT (system) ---')
print(SYSTEM_PROMPT)
print('\n--- PROMPT (user) ---')
print(build_user_prompt(low))
print('\n--- MODEL OUTPUT ---')
if USE_LLM:
    try:
        print(llm_explain(low))
    except Exception as e:
        print('LLM error:', e, '\nTemplate:', template_explanation(low))
else:
    print(template_explanation(low))

Lowest-risk claim: CCLM-00208  p=8%  Low
Drivers: total_billed=1,034.30 (lowers risk); payer_type=Commercial (lowers risk); payment_ratio=0.72 (lowers risk)

--- PROMPT (system) ---
You are a claims-denial analyst assistant. Write a short, plain-English risk explanation for ONE insurance claim. Follow every rule:
- Use ONLY the facts provided (probability, tier, and risk drivers). Do NOT invent facts, codes, names, or dollar amounts that are not given.
- Plain language an analyst can act on in seconds; no jargon or acronyms.
- Exactly 2-3 sentences.
- Include exactly ONE specific, concrete recommended action.
- Make clear this is a model risk ESTIMATE, not a guarantee of denial.

--- PROMPT (user) ---
Claim ID: CCLM-00208
Model denial probability: 8%
Risk tier: Low
Top risk drivers (from the model):
- total_billed = 1,034.30 (lowers denial risk)
- payer_type = Commercial (lowers denial risk)
- payment_ratio = 0.72 (lowers denial risk)

Write the explanation now.

--- MODEL OUTPUT ---
T